In [1]:
from pathlib import Path
import json

result_path = Path("../output")

json_files = list(result_path.glob('**/RunnerResult_DefaultRefiner.json'))
data = []
for file in json_files:
    try:
        datapoint = json.load(open(file))
        data.append(datapoint)
    except Exception as e:
        print(file)
len(data)

193

In [2]:
import pandas as pd

rows = []
for run in data:
    id = run['baseDir'].replace('/app/output/', '').replace('_', ':')
    perf = run.get("performanceTracker", {})

    for metric, events in perf.items():
        if not isinstance(events, list):
            continue
        for idx, ev in enumerate(events, start=1):
            rows.append({
                "id": id,
                "metric": metric,
                "attempt": idx,
                "start_time": ev.get("startTime"),
                "duration_ms": ev.get("duration"),
            })

perf_df = pd.DataFrame(rows)

# Optional convenience column
if not perf_df.empty:
    perf_df["duration_s"] = perf_df["duration_ms"] / 1000.0

perf_df

,id,metric,attempt,start_time,duration_ms,duration_s
0,npm:macaddress:20180511,codeql.init,1,1777881100927,9182,9.182
1,npm:macaddress:20180511,getExportsFromPackage,1,1777881110109,3143,3.143
2,npm:macaddress:20180511,model.query,1,1777881113265,630,0.630
3,npm:macaddress:20180511,model.query,2,1777881113912,1046,1.046
4,npm:macaddress:20180511,model.query,3,1777881114974,1900,1.900
...,...,...,...,...,...,...
8298,SNYK-JS-DOTTY-1577292,model.query,1,1777875341884,542,0.542
8299,SNYK-JS-DOTTY-1577292,model.query,2,1777875383241,10145,10.145
8300,SNYK-JS-DOTTY-1577292,model.query,3,1777875394103,12550,12.550
8301,SNYK-JS-DOTTY-1577292,codeql.analyse,1,1777875342454,40717,40.717


In [3]:
# Build a runtime summary by id, then append experiment-level totals
runtime_by_id_df = (
    perf_df.groupby("id", as_index=False)["duration_ms"]
    .sum()
    .rename(columns={"duration_ms": "total_runtime_ms"})
)

total_runtime_ms = runtime_by_id_df["total_runtime_ms"].sum()
num_ids = runtime_by_id_df["id"].nunique()
avg_runtime_per_id_ms = total_runtime_ms / num_ids if num_ids else 0

summary_rows_df = pd.DataFrame([
    {"id": "__TOTAL_EXPERIMENT__", "total_runtime_ms": total_runtime_ms},
    {"id": "__AVG_PER_ID__", "total_runtime_ms": avg_runtime_per_id_ms},
])

runtime_summary_df = pd.concat([runtime_by_id_df, summary_rows_df], ignore_index=True)

# Keep seconds for numeric analysis and add a human-readable duration string
runtime_summary_df["total_runtime_s"] = runtime_summary_df["total_runtime_ms"] / 1000.0
runtime_summary_df["total_runtime_human"] = pd.to_timedelta(
    runtime_summary_df["total_runtime_ms"], unit="ms"
).astype(str)

# Drop the millisecond column from final display
runtime_summary_df = runtime_summary_df.drop(columns=["total_runtime_ms"])

runtime_summary_df

,id,total_runtime_s,total_runtime_human
0,GHSA-6cj2-92m5-7mvp,235.983000,0 days 00:03:55.983000
1,GHSA-896r-f27r-55mw,136.138000,0 days 00:02:16.138000
2,GHSA-cf4h-3jhx-xvhq,316.883000,0 days 00:05:16.883000
3,GHSA-rp65-9cf3-cjxr,751.292000,0 days 00:12:31.292000
4,SNYK-JS-ALFREDWORKFLOWNODEJS-608975,259.780000,0 days 00:04:19.780000
...,...,...,...
190,npm:truncate:20180225,197.079000,0 days 00:03:17.079000
191,npm:valid-email:20180222,2024.498000,0 days 00:33:44.498000
192,npm:whereis:20180401,142.341000,0 days 00:02:22.341000
193,__TOTAL_EXPERIMENT__,203887.447000,2 days 08:38:07.447000
